# 00. 준비: 입력 고정, pool 구성, 운영 패리티, 임베딩, D2 원질문 선정

[계획서](../../../docs/ml/CACHE_EXPERIMENT.md) 3~5절을 실행한다. 이 노트북은 다음을 한 번에 봉인한다.

1. 입력 7개(stream, RouterBench 5-shot, xRouteBench train/test, 라우터 로그, 운영 프로브)의 SHA-256 확인
2. stream 5,368문항 + 운영 1.0 라우터 예측 + 선택된 단계 모델의 **기록 답변**을 묶은 pool
3. 채점기 자기 일치 검사: 각 기록 답변을 "다른 질문의 답"처럼 채점해 기록된 성공 여부와 비교
4. **운영 패리티 관문**: 2026-08-30 배포 스택이 반환한 73개 프로브의 코사인·리랭커 점수를 로컬이 재현해야 한다. 실패하면 멈춘다
5. BGE 임베딩(서비스처럼 한 문장씩, CPU)과 D2 원질문 300그룹 선정, 작성 시트 생성

Colab T4 기준 수 분(임베딩·라우팅 각 5,368문항). 로컬 CPU로 돌리면 1시간 가까이 걸린다. GCP·유료 API는 쓰지 않는다.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

RUN_ID = os.environ.get("CACHE_EXP_RUN_ID", "run001")  # 새 계획·재시작이면 새 ID
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    LLM_ROUTER = Path("/content/drive/MyDrive/LLM_Router")
    KIT = LLM_ROUTER / "cache-experiment"
    zip_path = KIT / "source/cache-source.zip"
    expected = (KIT / "source/cache-source.sha256").read_text().split()[0]
    assert hashlib.sha256(zip_path.read_bytes()).hexdigest() == expected, "source ZIP SHA mismatch"
    REPO = Path("/content/cache-source")
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
        z.extractall(REPO)
    for name, digest in json.loads((REPO / "source-bundle-manifest.json").read_text()).items():
        assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
    HERE = REPO / "ml/notebooks/cache_experiment"
    DATA_ROOT, OUTPUT_ROOT = KIT / "data-root", KIT / "runs"
    ROUTER_DIR = LLM_ROUTER / "transfer-experiments/inputs/experiment-A/models/r0"
    os.environ.setdefault("CACHE_EXP_DRIVE_LOGGER", str(LLM_ROUTER / "retraining-effect/logger/logger-log.parquet"))
    assert sys.version_info[:2] in {(3, 11), (3, 12)}, "런타임 버전 2026.07(Python 3.12)을 선택하세요"
    marker = REPO / ".installed-v2"
    if not marker.exists():
        # Text-only experiment. Colab's preinstalled vision stack does not match
        # torch 2.5.1, and transformers imports timm (which needs torchvision)
        # while resolving Auto classes, so remove all three.
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio", "timm"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.5.1",
                        "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(HERE / "requirements-colab.txt")], check=True)
        # Exercise the Auto-class resolution the notebook uses, in a fresh process,
        # so a preinstalled package clash fails here rather than after the restart.
        probe = ("from transformers import AutoConfig, AutoModel; "
                 "AutoModel.from_config(AutoConfig.from_pretrained('BAAI/bge-large-en-v1.5', "
                 "revision='d4aa6901d3a41ba39fb536a557fa166f842b0e09')); print('transformers ok')")
        subprocess.run([sys.executable, "-c", probe], check=True)
        marker.write_text("ok")
        # Colab already imported its own numpy; mixing it with the pinned one breaks
        # transformers. Restart the runtime (files in /content survive) and run all again.
        print("설치 완료. 런타임을 재시작합니다. 재시작되면 '모두 실행'을 다시 누르세요.")
        os.kill(os.getpid(), 9)
    import numpy
    pinned = next(l.split("==")[1].strip() for l in (HERE / "requirements-colab.txt").read_text().splitlines()
                  if l.startswith("numpy=="))
    assert numpy.__version__ == pinned, (
        f"numpy {numpy.__version__} != {pinned}: 런타임 → 세션 다시 시작 후 '모두 실행'을 다시 누르세요")
else:
    # 로컬: 저장소 루트의 .venv-cache 커널. 입력은 ml/data·ml/artifacts가 있는 체크아웃,
    # 산출물은 Drive 동기화 폴더의 kit runs/ 가 있으면 그곳(Colab과 같은 run을 이어 쓴다).
    HERE = Path.cwd()
    DATA_ROOT = Path(os.environ.get("CACHE_EXP_DATA_ROOT", HERE.parents[2]))
    synced = sorted(Path.home().glob("Library/CloudStorage/GoogleDrive-*/*/LLM_Router/cache-experiment/runs"))
    OUTPUT_ROOT = os.environ.get("CACHE_EXP_OUTPUT_ROOT") or (synced[0] if synced else None)
    ROUTER_DIR = None  # config router.package_path
    found = sorted(Path.home().glob("Library/CloudStorage/GoogleDrive-*/*/LLM_Router/retraining-effect/logger/logger-log.parquet"))
    if found:
        os.environ.setdefault("CACHE_EXP_DRIVE_LOGGER", str(found[0]))
CONFIG = Path(os.environ.get("CACHE_EXP_CONFIG", HERE / "experiment_config.json"))
sys.path.insert(0, str(HERE))

import pandas as pd
import cache_experiment as ce
import models
import pipeline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
run = pipeline.Run(DATA_ROOT, RUN_ID, CONFIG, OUTPUT_ROOT)
print("run directory:", run.root)

## 1. 라우터 로그 복사

재학습 효과 실험 00이 Drive에 남긴 `logger-log.parquet`(운영 1.0 anchor의 8192 추론)를 입력 위치로 복사한다. 이미 있으면 SHA만 확인한다. 경로는 첫 셀이 정한다.

In [ ]:
pipeline.copy_logger(run, os.environ.get("CACHE_EXP_DRIVE_LOGGER"))

## 2. 모델 적재

서빙과 같은 revision을 적재한다. `runtime_versions`는 torch 2.5.1 / transformers 4.48.3이 아니면 예외를 던진다. GPU에서 돌려도 아래 운영 패리티 관문이 서빙(CPU)과 같은 점수를 내는지 확인한다. 라우터는 운영 R0 파일 4개의 SHA로 확인한다(Colab은 Drive의 같은 사본).

In [ ]:
runtime = models.runtime_versions(run.config)  # 서빙 이미지와 torch/transformers 버전이 다르면 멈춘다
DEVICE = run.config["inference_device"]  # auto: Colab은 cuda. 운영 패리티 관문이 수치 일치를 확인한다
embedder = models.Embedder(run.config, device=DEVICE)
reranker = models.Reranker(run.config, device=DEVICE)
print(embedder.device, runtime)
router = models.Router(run.config, run.data_root, device=DEVICE, package_dir=ROUTER_DIR)
print("router:", router.path)

## 3. 준비 단계 실행과 봉인

라우터 관문: 로컬 라우터가 로거와 같은 입력(`model_input_text`) 200건에서 로거 판정을 99% 이상 재현해야 한다. 통과하면 서비스처럼 요청 원문으로 모두 분류한다. 서비스 입력 한도(BGE 512토큰)를 넘는 질문은 서비스가 거절하므로 pool에서 제외한다.

In [ ]:
manifest = pipeline.prepare(run, embedder, reranker, router, runtime)
{k: manifest[k] for k in ("pool", "exclusions", "scorer_selfcheck", "production_parity", "router_anchor_parity",
                          "router_input_diagnostic", "d2_originals", "d1_pool_rows")}

## 4. 확인

채점기 자기 일치 표와 D2 원질문 구성을 본다. 채점기가 기록과 다르게 판정한 행은 보고서의 한계로 남긴다(고치지 않는다).

In [ ]:
selfcheck = ce.read(run.path("prepare", "scorer-selfcheck.json"))
pd.DataFrame(selfcheck["table"])

In [ ]:
originals = run.d2_originals()
originals.groupby(["stratum", "d2_split"]).size().unstack(fill_value=0)

## 다음 단계

- `01_d1_replay.ipynb`는 지금 바로 실행할 수 있다.
- D2: `d2/authoring-sheet.csv`를 복사해 작성자가 `paraphrase_text`, `hard_negative_text`, `author`를 채운다. 작성 규칙은 `02_d2_sheets.ipynb` 맨 위에 있다.